# Validation benchmark (V1–V15)

## Utilities

In [ ]:
import json
import numpy as np
import pandas as pd
from models.clp_model import CLPModel
from settings import EXPERIMENTS_FOLDER, HYPERPARAMETERS_FOLDER

SETS = [f"V{k}" for k in range(1, 16)]
GROUPS = {"Avg. (V1–V7)": SETS[:7], "Avg. (V8–V15)": SETS[7:], "Avg. (V1–V15)": SETS}
SEEDS = [42, 43, 44]

def instance_vols(name):
    """Volume per (set, instance) from <name>_V{k}.csv."""
    vols = {}
    for s in SETS:
        df = pd.read_csv(EXPERIMENTS_FOLDER / f"{name}_{s}.csv")
        vols[s] = df.set_index("instance")[next(c for c in df.columns if c.startswith("Vol"))]
    return pd.concat(vols, names=["set", "instance"])

def seed_mean_vols(name):
    """Volume per (set, instance) averaged over the seeds. <name> contains '{seed}'."""
    return pd.concat([instance_vols(name.format(seed=seed)) for seed in SEEDS], axis=1).mean(axis=1)

def with_groups(table):
    """Appends the average rows of each group of sets."""
    return pd.concat([table, pd.DataFrame({g: table.loc[sets].mean() for g, sets in GROUPS.items()}).T])

def loss_gap(vol, ref):
    """Loss Gap (%) computed from the average volume of each row."""
    return ((100 - ref) - (100 - vol)) / (100 - ref) * 100

def set_means(name):
    """Mean volume per V set from <name>_V{k}.csv."""
    return instance_vols(name).groupby("set", sort=False).mean().rename(None)

def per_seed_gap(prefix, ref):
    """Loss Gap of each seed of <prefix>_S{seed}, per set and in the average rows."""
    return pd.DataFrame({f"S{seed}": loss_gap(with_groups(set_means(f"{prefix}_S{seed}").to_frame())[0], ref)
                         for seed in SEEDS})

def bold_max(row):
    return ["font-weight: bold" if v == row.max() else "" for v in row]

## Hyperparameter selection

In [ ]:
# Paired one-standard-error rule (Hastie et al.) over the instances of V1–V15 (seeds averaged):
#   - Candidates: variants whose mean difference with the best one does not exceed its paired SE.
#   - The candidate with the fewest parameters is selected; ties are broken by the highest mean.
# The variant selected at each step is the base configuration of the next one.
steps = {
    "DIM": ["DIM32", "DIM64", "DIM128"],
    "ENC": ["ENC1", "ENC4"],
    "CONT": ["CONT1", "CONT4"],
    "SCORING": ["SCORING1", "SCORING4"],
    "NHEAD": ["NHEAD1", "NHEAD4"],
    "DROPOUT": ["DROPOUT00", "DROPOUT02"],
}

def count_params(variant):
    with open(HYPERPARAMETERS_FOLDER / f"MIX_{variant}_S{SEEDS[0]}.json") as f:
        return sum(p.numel() for p in CLPModel(**json.load(f)).parameters())

def select(variants):
    vols = pd.DataFrame({v: seed_mean_vols(f"fast/bsg_hybrid_fast_MIX_{v}_S{{seed}}") for v in variants})
    best = vols.mean().idxmax()
    diffs = vols[best].to_numpy()[:, None] - vols.to_numpy()

    selection = pd.DataFrame({
        "Parameters": [count_params(v) for v in variants],
        "Mean": vols.mean().to_numpy(),
        "Gap to best": diffs.mean(axis=0),
        "Paired SE": diffs.std(axis=0, ddof=1) / np.sqrt(len(vols)),
    }, index=variants)
    selection["t"] = (selection["Gap to best"] / selection["Paired SE"]).where(selection.index != best)
    selection["Within 1 SE"] = selection["Gap to best"] <= selection["Paired SE"]

    candidates = selection[selection["Within 1 SE"]]
    chosen = candidates.sort_values(["Parameters", "Mean"], ascending=[True, False]).index[0]
    selection["Selected"] = selection.index == chosen
    return selection, chosen

tables = {}
base = None
for step, variants in steps.items():
    tables[step], base = select(variants if base is None else [base] + variants)
    print(f"{step:<8} -> {base}")

print(f"\nFinal choice: {base}")
pd.concat(tables, names=["Step", "Variant"]).round(4)

DIM      -> DIM64
ENC      -> ENC4
CONT     -> ENC4
SCORING  -> ENC4
NHEAD    -> ENC4
DROPOUT  -> ENC4

Final choice: ENC4


Parameters     Mean  Gap to best  Paired SE       t  \
Step    Variant                                                          
DIM     DIM32           39073  95.3848       0.0081     0.0072  1.1174   
        DIM64          151873  95.3871       0.0058     0.0063  0.9188   
        DIM128         598657  95.3929       0.0000     0.0000     NaN   
ENC     DIM64          151873  95.3871       0.0185     0.0071  2.6094   
        ENC1           143233  95.3788       0.0268     0.0077  3.4586   
        ENC4           184897  95.4056       0.0000     0.0000     NaN   
CONT    ENC4           184897  95.4056       0.0000     0.0000     NaN   
        CONT1          151425  95.3875       0.0181     0.0065  2.8043   
        CONT4          251841  95.3953       0.0103     0.0063  1.6374   
SCORING ENC4           184897  95.4056       0.0000     0.0000     NaN   
        SCORING1       159873  95.3916       0.0140     0.0066  2.1297   
        SCORING4       234945  95.4019       0.0037     0.0062  0.5926   
NHEAD   ENC4           184897  95.4056       0.0000     0.0000     NaN   
        NHEAD1         184897  95.4035       0.0022     0.0061  0.3540   
        NHEAD4         184897  95.3891       0.0165     0.0060  2.7682   
DROPOUT ENC4           184897  95.4056       0.0000     0.0000     NaN   
        DROPOUT00      184897  95.3855       0.0201     0.0076  2.6454   
        DROPOUT02      184897  95.3885       0.0171     0.0068  2.5026   

                   Within 1 SE  Selected  
Step    Variant                           
DIM     DIM32            False     False  
        DIM64             True      True  
        DIM128            True     False  
ENC     DIM64            False     False  
        ENC1             False     False  
        ENC4              True      True  
CONT    ENC4              True      True  
        CONT1            False     False  
        CONT4            False     False  
SCORING ENC4              True      True  
        SCORING1         False     False  
        SCORING4          True     False  
NHEAD   ENC4              True      True  
        NHEAD1            True     False  
        NHEAD4           False     False  
DROPOUT ENC4              True      True  
        DROPOUT00        False     False  
        DROPOUT02        False     False

## Median seed

In [ ]:
# Seed whose average volume on V1–V15 is the median of the three seeds
models = {"M-Mix": "MIX_ENC4", "Linear": "LINEAR"}

medians = {}
for name, prefix in models.items():
    seed_vols = pd.Series({seed: instance_vols(f"fast/bsg_hybrid_fast_{prefix}_S{seed}").mean() for seed in SEEDS}).sort_values()
    medians[name] = seed_vols.round(4).to_dict()
    print(f"{name}: median seed S{seed_vols.index[len(seed_vols) // 2]}")

pd.DataFrame(medians)

M-Mix: median seed S42
Linear: median seed S43


,M-Mix,Linear
43,95.3960,95.1601
42,95.4087,95.1579
44,95.4122,95.1620


## Model comparison

In [ ]:
# Loss Gap (%) w.r.t. BSG-VCS (w=8) of each model under BSG-Hybrid (w=8), averaged over 3 seeds
models = {"M3": "M3", "M30": "M30", "M100": "M100", "M-Mix": "MIX_ENC4"}

bsg_vcs = with_groups(set_means("fast/bsg_vcs_fast").to_frame())[0]
per_seed = {m: per_seed_gap(f"fast/bsg_hybrid_fast_{p}", bsg_vcs) for m, p in models.items()}
table = pd.DataFrame({m: v.mean(axis=1) for m, v in per_seed.items()})

print("Standard deviation of the Loss Gap across seeds:")
print(pd.DataFrame({m: v.loc[list(GROUPS)].std(axis=1) for m, v in per_seed.items()}).round(2).to_string())

# M-Mix vs M30 on V1–V15: number of seed pairs won by M-Mix
mix, m30 = per_seed["M-Mix"].loc["Avg. (V1–V15)"], per_seed["M30"].loc["Avg. (V1–V15)"]
print(f"\nV1–V15 | M-Mix: {mix.round(2).to_dict()} | M30: {m30.round(2).to_dict()}")
print(f"M-Mix outperforms M30 in {sum(a > b for a in mix for b in m30)}/{len(mix) * len(m30)} seed pairs\n")

table.style.format("{:.2f}").apply(bold_max, axis=1)

Standard deviation of the Loss Gap across seeds:
                 M3   M30  M100  M-Mix
Avg. (V1–V7)   0.40  0.38  0.09   0.23
Avg. (V8–V15)  0.12  0.20  0.10   0.31
Avg. (V1–V15)  0.20  0.26  0.09   0.18

V1–V15 | M-Mix: {'S42': 4.68, 'S43': 4.42, 'S44': 4.76} | M30: {'S42': 4.5, 'S43': 4.1, 'S44': 4.59}
M-Mix outperforms M30 in 7/9 seed pairs



,M3,M30,M100,M-Mix
V1,6.83,2.43,-0.41,7.16
V2,1.77,1.30,-1.55,2.72
V3,2.70,2.13,1.35,3.22
V4,0.78,1.61,1.51,1.45
V5,0.50,2.58,2.35,2.78
V6,0.69,3.39,3.93,3.32
V7,0.01,4.37,3.69,3.49
V8,2.62,6.73,6.18,5.47
V9,2.41,7.28,5.77,6.86
V10,1.54,5.83,5.97,5.14


## Loss weighting

In [ ]:
# Loss Gap (%) w.r.t. BSG-VCS (w=8) of M-Mix under four loss weighting schemes (averaged over 3 seeds), BSG-Hybrid (w=8)
configs = {
    "Hard-biased (.25/.25/.50)": "MIX_ENC4",
    "Medium-biased (.25/.50/.25)": "MIX_WMEDIUM",
    "Easy-biased (.50/.25/.25)": "MIX_WEASY",
    "Balanced (.33/.33/.33)": "MIX_WBAL",
}

bsg_vcs = with_groups(set_means("fast/bsg_vcs_fast").to_frame())[0]
per_seed = {c: per_seed_gap(f"fast/bsg_hybrid_fast_{p}", bsg_vcs) for c, p in configs.items()}
table = pd.DataFrame({c: v.mean(axis=1) for c, v in per_seed.items()})

print("Standard deviation of the Loss Gap across seeds:")
print(pd.DataFrame({c: v.loc[list(GROUPS)].std(axis=1) for c, v in per_seed.items()}).round(2).to_string())

# Hard-biased vs Balanced on V1–V15: number of seed pairs won by Hard-biased
hard = per_seed["Hard-biased (.25/.25/.50)"].loc["Avg. (V1–V15)"]
bal = per_seed["Balanced (.33/.33/.33)"].loc["Avg. (V1–V15)"]
print(f"\nV1–V15 | Hard-biased: {hard.round(2).to_dict()} | Balanced: {bal.round(2).to_dict()}")
print(f"Hard-biased outperforms Balanced in {sum(a > b for a in hard for b in bal)}/{len(hard) * len(bal)} seed pairs\n")

table.style.format("{:.2f}").apply(bold_max, axis=1)

Standard deviation of the Loss Gap across seeds:
               Hard-biased (.25/.25/.50)  Medium-biased (.25/.50/.25)  Easy-biased (.50/.25/.25)  Balanced (.33/.33/.33)
Avg. (V1–V7)                        0.23                         0.64                       0.37                    0.38
Avg. (V8–V15)                       0.31                         0.26                       0.47                    0.28
Avg. (V1–V15)                       0.18                         0.33                       0.40                    0.32

V1–V15 | Hard-biased: {'S42': 4.68, 'S43': 4.42, 'S44': 4.76} | Balanced: {'S42': 3.84, 'S43': 4.33, 'S44': 3.73}
Hard-biased outperforms Balanced in 9/9 seed pairs



,Hard-biased (.25/.25/.50),Medium-biased (.25/.50/.25),Easy-biased (.50/.25/.25),Balanced (.33/.33/.33)
V1,7.16,4.88,5.92,5.68
V2,2.72,3.54,2.48,3.18
V3,3.22,3.46,2.59,3.56
V4,1.45,0.87,0.73,1.44
V5,2.78,2.03,1.74,1.98
V6,3.32,2.07,3.26,3.26
V7,3.49,2.69,2.97,2.86
V8,5.47,4.21,4.11,4.91
V9,6.86,4.98,5.65,5.82
V10,5.14,3.89,3.56,3.59
